# Simple RAG Project (LlamaIndex + Hugging Face)
This notebook is a beginner-friendly RAG pipeline. It follows 5 simple steps:
1. Load documents
2. Split and embed text
3. Store vectors in an index
4. Retrieve relevant chunks
5. Generate an answer with an LLM

In [ ]:
# STEP 0 - Install dependencies (run once)
%pip install -q llama-index llama-index-llms-huggingface llama-index-embeddings-huggingface transformers accelerate bitsandbytes sentence-transformers pypdf

## Step 1: Import libraries
This step imports all required Python modules for loading files, creating embeddings, running the LLM, and building the RAG index.

In [ ]:
# STEP 1 - Imports
import os
from getpass import getpass
from pathlib import Path
import torch

from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings, PromptTemplate
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM

## Step 2: Enter your settings
You provide:
- the document path (file or folder)
- the LLM model to use
- generation token settings
- retrieval `top_k`
- optional Hugging Face token for gated models

In [ ]:
# STEP 2 - User inputs (simple and explicit)
print("RAG pipeline: Load document -> Split -> Embed -> Retrieve -> Generate answer")

LLM_OPTIONS = {
    "1": "microsoft/phi-3-mini-4k-instruct",
    "2": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "3": "meta-llama/Llama-2-7b-chat-hf"
}

print("\nChoose an LLM:")
for k, v in LLM_OPTIONS.items():
    print(f"  {k}. {v}")

doc_input = input("\nEnter document path (pdf/txt/docx) or folder path: " ).strip()
llm_choice = input("Choose model number [1]: " ).strip() or "1"
max_tokens_input = input("Max new tokens [256]: " ).strip() or "256"
temperature_input = input("Temperature [0.0]: " ).strip() or "0.0"
top_k_input = input("Retriever top_k [3]: " ).strip() or "3"

hf_token = getpass("Hugging Face token (optional, needed for gated models): " ).strip()

DOC_PATH = doc_input
MODEL_NAME = LLM_OPTIONS.get(llm_choice, LLM_OPTIONS["1"])
MAX_NEW_TOKENS = int(max_tokens_input)
TEMPERATURE = float(temperature_input)
TOP_K = int(top_k_input)

print("\nSelected config:")
print(f"  document: {DOC_PATH}")
print(f"  model: {MODEL_NAME}")
print(f"  max_new_tokens: {MAX_NEW_TOKENS}")
print(f"  temperature: {TEMPERATURE}")
print(f"  top_k: {TOP_K}")

## Step 3: Token setup (optional)
If you select a gated/private model, add your Hugging Face token here. For public models, you can leave it empty.

In [ ]:
# STEP 3 - Optional token setup
if hf_token:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token
    os.environ["HF_TOKEN"] = hf_token
    print("HF token saved in current notebook session.")
else:
    print("No HF token provided. Public models should still work.")

## Step 4: Load your document(s)
This step reads your input path and loads the text content into memory. It supports a single file or a folder of files.

In [ ]:
# STEP 4 - Load and parse documents
path_obj = Path(DOC_PATH).expanduser()
if not path_obj.exists():
    raise FileNotFoundError(f"Path does not exist: {path_obj}")

if path_obj.is_file():
    documents = SimpleDirectoryReader(input_files=[str(path_obj)]).load_data()
else:
    documents = SimpleDirectoryReader(input_dir=str(path_obj), recursive=True).load_data()

print(f"Loaded {len(documents)} document chunk(s).")

## Step 5: Create the LLM and embedding model
RAG needs two models:
- LLM: generates final answers
- Embedding model: converts text into vectors for retrieval

In [ ]:
# STEP 5 - Create prompts and models
system_prompt = (
    "You are a helpful RAG assistant. ",
    "Answer with facts from the retrieved context. ",
    "If context is insufficient, say you are not sure."
)

query_wrapper_prompt = PromptTemplate(
    "<|user|>{query_str}</s><|assistant|>"
)

use_cuda = torch.cuda.is_available()
dtype = torch.float16 if use_cuda else torch.float32
model_kwargs = {"torch_dtype": dtype}
if use_cuda:
    model_kwargs["load_in_8bit"] = True

llm = HuggingFaceLLM(
    model_name=MODEL_NAME,
    tokenizer_name=MODEL_NAME,
    context_window=4096,
    max_new_tokens=MAX_NEW_TOKENS,
    generate_kwargs={"temperature": TEMPERATURE, "do_sample": TEMPERATURE > 0},
    system_prompt="".join(system_prompt),
    query_wrapper_prompt=query_wrapper_prompt,
    device_map="auto" if use_cuda else None,
    model_kwargs=model_kwargs,
)

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("LLM and embedding models are ready.")

## Step 6: Configure chunking and global settings
The document is split into chunks before indexing. Smaller chunks improve precision; larger chunks add more context.

In [ ]:
# STEP 6 - Configure global RAG settings
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512
Settings.chunk_overlap = 50
print("Settings applied: chunk_size=512, chunk_overlap=50")

## Step 7: Build the vector index
This is the core RAG indexing step. Your chunk embeddings are stored so the retriever can find relevant context at query time.

In [ ]:
# STEP 7 - Build vector index (embedding + storage)
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=TOP_K)
print("Index built. RAG is ready for questions.")

## Step 8: Ask questions and inspect retrieval
The helper function below prints:
- the final generated answer
- retrieved chunks used as evidence (with similarity score)
This makes the RAG process transparent and instructive.

In [ ]:
# STEP 8 - Ask function (shows answer + retrieved evidence)
def ask_rag(question: str):
    response = query_engine.query(question)
    print("\n=== Final Answer ===")
    print(str(response))

    if hasattr(response, "source_nodes") and response.source_nodes:
        print("\n=== Retrieved Chunks (RAG retrieve step) ===")
        for i, node in enumerate(response.source_nodes, start=1):
            score = getattr(node, "score", None)
            text = node.node.get_text().replace("\n", " ")[:300]
            print(f"[{i}] score={score}")
            print(text)
            print("-" * 60)
    else:
        print("No source nodes available in response.")

    return response

## Step 9: Run a sample question
This is a quick test to verify the full pipeline works.

In [ ]:
# STEP 9 - Example query
example_question = "Summarize the document in simple words."
_ = ask_rag(example_question)

## Step 10: Ask your own question
Edit the question text and run the cell again as many times as you want.

In [ ]:
# STEP 10 - Your own query
# Change the text below and run this cell again.
my_question = "Write your question here"
_ = ask_rag(my_question)